Исследование датасета Dataset_22022022

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from artifact_loader import original_data_path

In [2]:
Path.cwd().name == "notebooks"

True

In [3]:
original_data_path()

PosixPath('/Users/l.tereshchenkova/MIPT/startup/barko-docs/analytics/research/datasets/foreign/bohland_2023_shelter_behavior_after_adoption_supplement_s4.csv')

In [4]:
df_cbarq = pd.read_csv(original_data_path())

In [6]:
list(df_cbarq.columns)

['Record ID',
 'Event Name',
 'Repeat Instrument',
 'Repeat Instance',
 'Survey Identifier',
 'Survey Timestamp',
 "Providing consent:    By clicking Accept I, [consent_owner_name], provide consent to the following:     I am aware that I am being asked to participate in a clinical trial entitled 'Identifying Behavior Changes in Dogs During the Six Months Following Adoption from a Shelter.'   I have had the opportunity to ask questions and have had them answered to my satisfaction.    I voluntarily agree to participate in this study. I am not giving up any legal rights by agreeing to participate.   I have read the information about the clinical trial, and understand the purpose and requirements.   I give my permission to publish data obtained from this study. I understand that all personal identifying information will be kept confidential.  I grant permission to contact me with information regarding trial updates and future clinical trials.         ",
 'Date:',
 'Please provide the name

In [8]:
df_cbarq_10 = df_cbarq[df_cbarq["Event Name"] == "10 days after adoption"].copy()


In [14]:
df_cbarq_10.dropna(axis=1, how='all', inplace=True)
df_cbarq_10.shape

(99, 111)

In [16]:
list(df_cbarq_10.columns)

['Record ID',
 'Event Name',
 'Survey Timestamp.2',
 'Date/Time Completed:',
 'Dog shelter name: ',
 'Animal ID:   You can find this number in the documents provided by the shelter . ',
 'Just before being taken for a walk:',
 'Just before being taken on a car trip:',
 'When approached directly by an unfamiliar person while being walked/exercised on a leash:',
 'When toys, bones or other objects are taken away by a household member:',
 'When approached directly by a household member while s/he (the dog) is eating:',
 'When mailmen or other delivery workers approach your home:',
 'When his/her food is taken away by a household member:',
 'When approached directly by an unfamiliar dog while being walked/exercised on a leash:',
 'When strangers walk past your home when your dog is outside or in the yard:',
 'When barked, growled, or lunged at by another (unfamiliar) dog:',
 'When approached while eating by another (familiar) household dog (leave blank if no other dogs):',
 'When approache

In [ ]:
new_columns = [
    # --- demographics & service (16) ---
    "dog_age_years", "dog_age_months", "breed", "breed_other", "dog_sex",
    "neutered_spayed", "age_at_adoption_years", "age_at_adoption_months",
    "owner_sex", "owner_age", "owner_profession", "owner_profession_other",
    "other_dogs_present", "other_dogs_by_age", "other_dogs_by_age_a", "other_dogs_by_age_b",
    # --- aggression Q1–Q35 ---
    *[f"question_{i:03d}" for i in range(1, 36)],
    "other_aggression_present", "other_aggression_describe",
    # --- fear Q36–Q53 ---
    *[f"question_{i:03d}" for i in range(36, 54)],
    # --- separation Q54–Q61 ---
    *[f"question_{i:03d}" for i in range(54, 62)],
    "other_fear_present", "other_fear_describe",
    # --- excitability Q62–Q67 ---
    *[f"question_{i:03d}" for i in range(62, 68)],
    "other_excitability_present", "other_excitability_describe",
    # --- attachment + miscellaneous Q68–Q100 ---
    *[f"question_{i:03d}" for i in range(68, 101)],
    "other_bizarre_behavior_describe",
]

assert len(new_columns) == df_cbarq.shape[1], (len(new_columns), df_cbarq.shape[1])
df_cbarq.columns = new_columns

In [ ]:
na_counts = df_cbarq.isna().sum()
na_counts[na_counts > 0].sort_values(ascending=False)

In [ ]:
# df_cbarq_original = pd.read_csv(PATHS["original_data"])

In [ ]:
df_cbarq[df_cbarq['breed'].isna()]

In [ ]:
df_cbarq.dropna(subset=['breed'], inplace=True)

In [ ]:
df_cbarq.info()

In [ ]:
list(df_cbarq.columns)

In [ ]:
na_counts = df_cbarq.isna().sum()
na_counts[na_counts > 0].sort_values(ascending=False)

In [ ]:
df_cbarq.drop(columns=['breed_other',
'other_bizarre_behavior_describe',
'other_aggression_describe',
'owner_profession_other',
'other_excitability_describe',
'other_fear_describe',
'other_dogs_by_age_a',
'other_dogs_by_age',
'other_dogs_by_age_b',
'question_032',
'question_033',
'question_034',
'question_035'], inplace=True)

In [ ]:
na_counts = df_cbarq.isna().sum()
na_counts[na_counts > 0].sort_values(ascending=False)

In [ ]:
df_cbarq['age_at_adoption_years'].value_counts()

In [ ]:
df_cbarq.loc[df_cbarq['age_at_adoption_years'].isna(), 'age_at_adoption_years'] = 0

In [ ]:
df_cbarq['age_at_adoption_months'] = (
    df_cbarq['age_at_adoption_months']
    .str.replace(',', '.', regex=False)
    .astype('float')
)
df_cbarq.loc[df_cbarq['age_at_adoption_months'].isna(), 'age_at_adoption_months'] = 0
df_cbarq = df_cbarq.astype({'age_at_adoption_years': 'int32', 'age_at_adoption_months': 'int32'})

In [ ]:
# 999
# диапазон ответов на вопросы

df_cbarq.describe()
for i in range(1,101):
    try:
        print(df_cbarq[f'question_{i:03d}'].value_counts())
    except:
        print(f'question_{i:03d} error')

for i in range(1,101):
    try:
        df_cbarq.loc[df_cbarq['question_{i:03d}'] == 999, 'question_{i:03d}'] = None
    except:
        print(f'question_{i:03d} error')
for i in range(1,101):
    try:
        df_cbarq.dropna(subset=['question_{i:03d}'], inplace=True)
    except:
        print(f'question_{i:03d} error')



In [ ]:
# выбросы по возрасту
sns.boxplot(data=df_cbarq[['dog_age_months']])
plt.show()
display(df_cbarq[df_cbarq['dog_age_months'] > 12][['dog_age_months', 'dog_age_years']])
display(df_cbarq[df_cbarq['dog_age_years'] == 1][['dog_age_months', 'dog_age_years']])

In [ ]:
df_cbarq.drop(list(df_cbarq[df_cbarq['dog_age_months'] == 15].index), inplace=True)

In [ ]:
df_cbarq.loc[list(df_cbarq[df_cbarq['dog_age_months'] > 12].index), 'dog_age_months'] = df_cbarq['dog_age_months'] % 12

In [ ]:
# выбросы по возрасту владельца
sns.boxplot(data=df_cbarq[['owner_age']])
plt.show()


In [ ]:
# выбросы по профессии владельца
sns.histplot(data=df_cbarq['owner_profession'], )
plt.show()

In [ ]:
sns.histplot(data=df_cbarq['dog_sex'], )
plt.show()

In [ ]:
sns.histplot(data=df_cbarq['owner_sex'])
plt.show()

In [ ]:
df_cbarq.drop(columns=['owner_sex'], inplace=True)

In [ ]:
display(df_cbarq[['dog_age_years',
 'dog_age_months',
 'breed',
 'dog_sex',
 'neutered_spayed',
 'age_at_adoption_years',
 'age_at_adoption_months',
 'owner_age',
 'owner_profession',
 'other_dogs_present', 
 'other_aggression_present', 
 'other_fear_present', 
 'other_excitability_present']].head())


df_cbarq['breed'].value_counts()

In [ ]:
df_cbarq = df_cbarq.astype({'dog_age_years': 'int', 'dog_age_months': 'int', 'neutered_spayed': 'int', 'owner_age': 'int', 'other_dogs_present': 'int'})
df_cbarq['neutered_spayed'].dtype

In [ ]:
df_cbarq_encoded = pd.get_dummies(df_cbarq, columns=['dog_sex'], dtype=int)
df_cbarq_encoded['neutered_spayed'].dtype

In [ ]:
df_cbarq_encoded['neutered_spayed'].value_counts()

In [ ]:
df_cbarq_encoded.loc[df_cbarq_encoded['neutered_spayed'] == 999, 'neutered_spayed'] = None
df_cbarq_encoded.dropna(subset=['neutered_spayed'], inplace=True)
df_cbarq_encoded['neutered_spayed'] = df_cbarq_encoded['neutered_spayed'].astype('int')

# выбросы по количеству собак по возрасту и полу
# выбросы по количеству собак по возрасту и полу и породе

In [ ]:
df_cbarq_encoded['neutered_spayed'].dtype

In [ ]:
sns.histplot(data=df_cbarq_encoded['neutered_spayed'])
plt.show()

In [ ]:
# age_at_adoption_years',
#  'age_at_adoption_months'
df_cbarq_encoded[['age_at_adoption_months']].value_counts()
df_cbarq_encoded.loc[df_cbarq_encoded['age_at_adoption_years'] > df_cbarq_encoded['dog_age_years']].value_counts()



In [ ]:
# other_dogs_present', 
#  'other_aggression_present', 
#  'other_fear_present', 
#  'other_excitability_present
sns.histplot(data=df_cbarq_encoded['other_dogs_present'])
plt.show()
sns.histplot(data=df_cbarq_encoded['other_aggression_present'])
plt.show()
sns.histplot(data=df_cbarq_encoded['other_fear_present'])
plt.show()
sns.histplot(data=df_cbarq_encoded['other_excitability_present'])
plt.show()

In [ ]:
sns.pairplot(data=df_cbarq_encoded[['dog_age_years',
 'dog_age_months',
 'breed',
 'neutered_spayed',
 'age_at_adoption_years',
 'age_at_adoption_months',
 'owner_age',
 'owner_profession',
 'other_dogs_present', 
 'other_aggression_present', 
 'other_fear_present', 
 'other_excitability_present']])

plt.show()

In [ ]:
breeds = df_cbarq_encoded['breed'].value_counts()

In [ ]:
breeds.index = breeds.index.str.lower()
breeds.index = breeds.index.str.replace(' ', '_')
breeds.index = breeds.index.str.replace('-', '_')
breeds.index = breeds.index.str.replace('/', '_')
breeds.index = breeds.index.str.replace('(', '')
breeds.index = breeds.index.str.replace(')', '')
breeds.index = breeds.index.str.replace('.', '')

In [ ]:
breeds.index

In [ ]:
BREED_GROUP = {
  "incrocio___meticcio": "mixed",
  "altro": "other",

  # herding
  "border_collie": "herding",
  "pastore_tedesco": "herding",
  "australian_shepherd": "herding",
  "pastore_scozzese_shetland": "herding",
  "pastore_australiano_kelpie": "herding",
  "pastore_maremmano_abruzzese": "herding",
  "pastore_belga_malinois": "herding",
  "pastore_belga_groenendael": "herding",
  "pastore_belga_tervueren": "herding",
  "cane_da_pastore_scozzese_a_pelo_lungo": "herding",
  "welsh_corgi_cardigan": "herding",

  # gundog
  "labrador_retriever": "gundog",
  "golden_retriever": "gundog",
  "cocker_spaniel_inglese": "gundog",
  "cocker_americano": "gundog",
  "setter_inglese": "gundog",
  "setter_irlandese": "gundog",
  "pointer_inglese": "gundog",
  "springer_spaniel_inglese": "gundog",
  "weimaraner_a_pelo_corto": "gundog",
  "bracco_italiano": "gundog",
  "bracco_ungherese_a_pelo_corto": "gundog",
  "epagneul_breton": "gundog",
  "flat_coated_retriever": "gundog",
  "lagotto_romagnolo": "gundog",

  # terrier
  "jack_russel_terrier": "terrier",
  "parson_jack_russel_terrier": "terrier",
  "border_terrier": "terrier",
  "yorkshire_terrier": "terrier",
  "boston_terrier": "terrier",
  "staffordshire_bull_terrier": "terrier",

  # molosser
  "boxer": "molosser",
  "bulldog": "molosser",
  "bouledogue_francese": "molosser",
  "cane_corso": "molosser",
  "rottweiler": "molosser",
  "dobermann": "molosser",
  "dogo_argentino": "molosser",
  "alano": "molosser",
  "american_staffordshire_terrier": "molosser",
  "american_pitbull_terrier": "molosser",

  # spitz / nordic
  "siberian_husky": "spitz_nordic",
  "alaskan_malamute": "spitz_nordic",
  "samoiedo": "spitz_nordic",
  "akita": "spitz_nordic",
  "akita_americano_grande_cane_giapponese": "spitz_nordic",
  "shiba_inu": "spitz_nordic",
  "spitz_tedesco_nano": "spitz_nordic",
  "volpino_italiano": "spitz_nordic",
  "eurasier": "spitz_nordic",
  "cane_da_lupo_cecoslovacco": "spitz_nordic",
  "laika_russo_europeo": "spitz_nordic",

  # scenthound
  "beagle": "scenthound",
  "beagle_harrier": "scenthound",
  "bloodhound_chien_de_saint_hubert": "scenthound",
  "segugio_italiano_a_pelo_raso": "scenthound",
  "griffon_bleu_de_gascogne": "scenthound",
  "bassotto_tedesco_nano_a_pelo_corto": "scenthound",
  "bassotto_tedesco_standard_a_pelo_corto": "scenthound",
  "bassotto_tedesco_kaninchen_a_pelo_corto": "scenthound",
  "bassotto_tedesco_nano_a_pelo_duro": "scenthound",
  "bassotto_tedesco_standard_a_pelo_duro": "scenthound",

  # sighthound
  "whippet": "sighthound",
  "galgo_espanol": "sighthound",
  "piccolo_levriero_italiano": "sighthound",
  "cirneco_dell’etna": "sighthound",

  # companion / toy
  "maltese": "companion_toy",
  "chihuahua_a_pelo_corto": "companion_toy",
  "chihuahua_a_pelo_lungo": "companion_toy",
  "shih_tzu": "companion_toy",
  "cavalier_king_charles_spaniel": "companion_toy",
  "bichon_a_poil_frise": "companion_toy",
  "barbone_nano": "companion_toy",
  "barbone_toy": "companion_toy",
  "barbone_media_mole": "companion_toy",
  "pinscher_nano": "companion_toy",

  # working
  "hovawart": "working",
  "bovaro_del_bernese": "working",
  "terranova": "working",
  "rhodesian_ridgeback": "working",
  "basenji": "working",

  # pinscher / schnauzer
  "pinscher": "pinscher_schnauzer",
  "schnauzer_nano_zwergschnauzer": "pinscher_schnauzer",
  "schnauzer_medio": "pinscher_schnauzer",
}

BRACHYCEPHALIC_BREEDS = {
  "bulldog",
  "bouledogue_francese",
  "boston_terrier",
  "boxer",
  "cavalier_king_charles_spaniel",
  "shih_tzu",
}

def normalize_breed(s: str) -> str:
    return (
        s.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "")
    )

df_cbarq_encoded["breed_norm"] = df_cbarq_encoded["breed"].map(normalize_breed)
df_cbarq_encoded["breed_group"] = df_cbarq_encoded["breed_norm"].map(BREED_GROUP).fillna("other")
df_cbarq_encoded["is_mixed"] = (df_cbarq_encoded["breed_group"] == "mixed").astype("int8")
df_cbarq_encoded["is_brachycephalic"] = df_cbarq_encoded["breed_norm"].isin(BRACHYCEPHALIC_BREEDS).astype("int8")

In [ ]:
top_breeds = df_cbarq_encoded["breed_norm"].value_counts().head(10).index
df_cbarq_encoded["breed_top"] = np.where(
    df_cbarq_encoded["breed_norm"].isin(top_breeds),
    df_cbarq_encoded["breed_norm"],
    "other_rare",
)
# Использовать breed_group + breed_top (11–12 категорий)

In [ ]:
df_cbarq_clean = df_cbarq_encoded.copy()

In [ ]:
df_cbarq_clean = pd.get_dummies(df_cbarq_clean, columns=['breed_top', 'breed_group'], dtype=int)


In [ ]:
df_cbarq_clean.drop(columns=['breed', 'owner_profession'], inplace=True)

In [ ]:
df_cbarq_clean.drop(columns=['breed_norm'], inplace=True)

### Шаг 1. Очистка item-level ответов C-BARQ

- `999` → `NaN` (не наблюдалось / не применимо)
- значения вне 0–4 → `NaN` (артефакты ввода, напр. `803` в битых строках)

In [ ]:
NA_SENTINEL = 999
VALID_MIN, VALID_MAX = 0, 4


def clean_questions(df, question_cols=None, inplace=False):
    """Очистка ответов C-BARQ: 999→NaN, валидный диапазон 0–4."""
    out = df if inplace else df.copy()
    if question_cols is None:
        question_cols = [c for c in out.columns if c.startswith("question_")]

    out[question_cols] = out[question_cols].apply(pd.to_numeric, errors="coerce")
    out[question_cols] = out[question_cols].replace(NA_SENTINEL, np.nan)

    invalid = out[question_cols].notna() & (
        (out[question_cols] < VALID_MIN) | (out[question_cols] > VALID_MAX)
    )
    out[question_cols] = out[question_cols].mask(invalid)

    return out




In [ ]:
question_cols = [c for c in df_cbarq_clean.columns if c.startswith("question_")]
na_before = (df_cbarq_clean[question_cols] == NA_SENTINEL).sum().sum()
na_before

In [ ]:
numeric_questions = df_cbarq_clean[question_cols].apply(pd.to_numeric, errors="coerce")
out_of_range_before = (
    numeric_questions.notna()
    & ((numeric_questions < VALID_MIN) | (numeric_questions > VALID_MAX))
).sum().sum()



In [ ]:
df_cbarq_clean = clean_questions(df_cbarq_clean)

print(f"999 replaced: {na_before}")
print(f"out-of-range → NaN: {out_of_range_before}")
print(f"remaining 999: {(df_cbarq_clean[question_cols] == NA_SENTINEL).sum().sum()}")
print(f"value range: {df_cbarq_clean[question_cols].min().min():.0f} – {df_cbarq_clean[question_cols].max().max():.0f}")
print(f"NaN share: {df_cbarq_clean[question_cols].isna().mean().mean():.1%}")

### Шаг 2. Reverse-coding для trainability (Q5–Q7)

Пункты с негативной формулировкой («медленно учится», «легко отвлекается») инвертируются: `4 - value`, чтобы все 8 пунктов trainability измеряли одно направление (выше = лучше послушание).

- Q5 — медленно реагирует на коррекцию
- Q6 — медленно учится
- Q7 — легко отвлекается

Q1–Q4 и Q8 не инвертируются.

In [ ]:
TRAINABILITY_POSITIVE_ITEMS = [f"question_{i:03d}" for i in range(1, 5)] + ["question_008"]
TRAINABILITY_REVERSE_ITEMS = ["question_005", "question_006", "question_007"]
TRAINABILITY_ALL_ITEMS = TRAINABILITY_POSITIVE_ITEMS + TRAINABILITY_REVERSE_ITEMS


def reverse_trainability_items(df, inplace=False):
    """Инвертировать негативные пункты trainability: x → 4 - x."""
    out = df if inplace else df.copy()
    for col in TRAINABILITY_REVERSE_ITEMS:
        if col not in out.columns:
            continue
        mask = out[col].notna()
        out.loc[mask, col] = VALID_MAX - out.loc[mask, col]
    return out


def trainability_corr_matrix(df, items, label):
    """Средняя попарная корреляция внутри домена trainability."""
    sub = df[items].dropna(how="any")
    if len(sub) < 10:
        return pd.DataFrame(), np.nan
    corr = sub.corr()
    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    mean_r = corr.where(mask).stack().mean()
    print(f"{label}: n={len(sub)}, mean pairwise r={mean_r:.3f}")
    return corr, mean_r


# Сохраняем копию до reverse для проверки
df_before_reverse = df_cbarq_clean.copy()

# Корреляции до reverse (смешанное направление пунктов)
corr_before, r_before = trainability_corr_matrix(
    df_before_reverse, TRAINABILITY_ALL_ITEMS, "Before reverse"
)
display(corr_before.round(2))

# Применяем reverse-coding
df_cbarq_clean = reverse_trainability_items(df_cbarq_clean)

# После reverse: все пункты кодированы в направлении «выше = лучше trainability»
corr_after, r_after = trainability_corr_matrix(
    df_cbarq_clean, TRAINABILITY_ALL_ITEMS, "After reverse"
)
display(corr_after.round(2))

# Контроль: r(Q1, Q5) должен сменить знак (преобразование x → 4−x)
r_q1_q5_before = df_before_reverse[["question_001", "question_005"]].dropna().corr().iloc[0, 1]
r_q1_q5_after = df_cbarq_clean[["question_001", "question_005"]].dropna().corr().iloc[0, 1]
print(f"r(Q1, Q5): {r_q1_q5_before:.3f} → {r_q1_q5_after:.3f} (знак должен инвертироваться)")

# Контроль: Q5–Q7 до/после
for col in TRAINABILITY_REVERSE_ITEMS:
    before = df_before_reverse[col].dropna()
    after = df_cbarq_clean.loc[before.index, col]
    print(
        f"{col}: mean {before.mean():.2f} → {after.mean():.2f}, "
        f"r(before, after)={before.corr(after):.3f} (ожидается −1)"
    )

### Шаг 3. Агрегация item → domain mean

Сырые ответы (`question_*`) сворачиваются в **13 доменных шкал C-BARQ** по среднему `mean(skipna)` — стандартный способ scoring.

- Домен считается, если ответлен ≥ `MIN_ITEMS` пунктов из домена

In [ ]:
DOMAIN_ITEMS = {
    "trainability": [f"question_{i:03d}" for i in range(1, 9)],
    "stranger_directed_aggression": [
        "question_010", "question_011", "question_015", "question_016", "question_021", "question_028",
    ],
    "owner_directed_aggression": [
        "question_009", "question_013", "question_017", "question_019", "question_030", "question_031",
    ],
    "dog_directed_aggression": ["question_023", "question_024", "question_029"],
    "stranger_directed_fear": ["question_036", "question_037", "question_039", "question_040"],
    "nonsocial_fear": [
        "question_038", "question_041", "question_042", "question_044", "question_047", "question_048",
    ],
    "dog_directed_fear": ["question_045", "question_046"],
    "touch_sensitivity": ["question_049", "question_050", "question_051"],
    "separation_related": [f"question_{i:03d}" for i in range(54, 62)],
    "excitability": [f"question_{i:03d}" for i in range(62, 68)],
    "attachment": [f"question_{i:03d}" for i in range(68, 74)],
    "chasing": ["question_074", "question_075", "question_076"],
    "energy": ["question_090", "question_091", "question_092"],
}

MIN_ITEMS = 1


def to_domain_scores(df, domain_items=DOMAIN_ITEMS, min_items=MIN_ITEMS):
    """Агрегировать item-level ответы в доменные шкалы C-BARQ."""
    scores = {}
    coverage = {}
    for name, items in domain_items.items():
        items = [i for i in items if i in df.columns]
        if not items:
            scores[name] = pd.Series(np.nan, index=df.index)
            coverage[name] = pd.Series(np.nan, index=df.index)
            continue
        sub = df[items]
        n_valid = sub.notna().sum(axis=1)
        mean = sub.mean(axis=1, skipna=True)
        scores[name] = mean.where(n_valid >= min_items)
        coverage[name] = n_valid / len(items)
    return pd.DataFrame(scores), pd.DataFrame(coverage)


df_domains, df_domain_coverage = to_domain_scores(df_cbarq_clean)

print(f"rows: {len(df_domains)}, domains: {df_domains.shape[1]}")
print(f"domain NaN rate: {df_domains.isna().mean().mean():.2%}")
print(f"rows all domains complete: {(~df_domains.isna().any(axis=1)).mean():.1%}")
print("\nPer-domain missing (score undefined):")
display(df_domains.isna().mean().sort_values().to_frame("nan_rate").round(3))
print("\nDomain score summary:")
display(df_domains.describe().round(2))

### Шаг 4. Фильтр строк

Убираем собак с недостаточным покрытием доменных шкал — строки, где заполнено < `MIN_DOMAINS` из 13 доменов. Индексы `df_cbarq_clean` синхронизируются с `df_domains`.

In [ ]:
MIN_DOMAINS = 13


def filter_rows_by_domains(df_domains, df_domain_coverage, df_meta, min_domains=MIN_DOMAINS):
    """Оставить строки с ≥ min_domains заполненными доменными шкалами."""
    n_domains_ok = df_domains.notna().sum(axis=1)
    keep = n_domains_ok >= min_domains
    dropped = ~keep

    report = {
        "total": len(df_domains),
        "kept": int(keep.sum()),
        "dropped": int(dropped.sum()),
        "kept_pct": keep.mean(),
    }
    return keep, n_domains_ok, report


keep_mask, n_domains_ok, filter_report = filter_rows_by_domains(
    df_domains, df_domain_coverage, df_cbarq_clean, min_domains=MIN_DOMAINS,
)

print(
    f"MIN_DOMAINS={MIN_DOMAINS}: kept {filter_report['kept']} / {filter_report['total']} "
    f"({filter_report['kept_pct']:.1%}), dropped {filter_report['dropped']}"
)

# Распределение числа заполненных доменов
sns.histplot(n_domains_ok, bins=range(0, df_domains.shape[1] + 2), discrete=True)
plt.axvline(MIN_DOMAINS - 0.5, color="red", linestyle="--", label=f"threshold={MIN_DOMAINS}")
plt.xlabel("filled domains")
plt.title("Domain coverage per dog")
plt.legend()
plt.show()

if filter_report["dropped"] > 0:
    dropped_idx = df_domains.index[~keep_mask]
    missing_domains = (~df_domains.loc[dropped_idx].notna()).sum()
    print("\nDropped rows — missing domains (count across dropped):")
    display(missing_domains.sort_values(ascending=False).to_frame("n_rows"))
    print("\nDropped rows detail:")
    display(
        pd.DataFrame({
            "n_domains_filled": n_domains_ok.loc[dropped_idx],
            "mean_coverage": df_domain_coverage.loc[dropped_idx].mean(axis=1),
        }).sort_values("n_domains_filled")
    )

# Применяем фильтр
df_domains = df_domains.loc[keep_mask].copy()
df_domain_coverage = df_domain_coverage.loc[keep_mask].copy()
df_cbarq_clean = df_cbarq_clean.loc[keep_mask].copy()

print(f"\nAfter filter: {len(df_domains)} rows, domain NaN rate: {df_domains.isna().mean().mean():.2%}")
print(f"rows all domains complete: {(~df_domains.isna().any(axis=1)).mean():.1%}")

In [ ]:
for key, df in {
    "df_domains": df_domains,
    "df_cbarq_clean": df_cbarq_clean,
}.items():
    out = PATHS[key]
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    print(
        f"Saved: {out} — "
        f"{df.shape[0]} rows, {df.shape[1]} columns"
    )


In [ ]:
out = PATHS["domain_items"]
out.parent.mkdir(parents=True, exist_ok=True)

with open(out, "w", encoding="utf-8") as f:
    json.dump(DOMAIN_ITEMS, f, indent=2, ensure_ascii=False)

print(f"Saved: {out} — {len(DOMAIN_ITEMS)} domains")